# XP Phase 2 spike (throwaway): MiniGrid DoorKey timing and size floor
Runs PPO (Kairi-style) and PPO+RND (Kaiser-style) on DoorKey-6x6, then evaluates zero-shot at sizes 5/6/8/12 against a random policy.
Answers: how long does a run take, and are sizes 8 and 12 stuck at 0% success?

Use a CPU runtime (the env is CPU-bound). Results are saved to Drive so a disconnect loses nothing: rerun and finished runs are skipped.
Does not install the project's pinned numpy, so no runtime restart should be needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/xp-spike'
STEPS = 500_000   # per run; lower (e.g. 100_000) for a quick timing-only pass
SEED = 0
!mkdir -p $OUT

In [ ]:
REPO_URL = 'https://github.com/knguyen0923/xp-agents'
!rm -rf xp-agents && git clone --depth 1 $REPO_URL xp-agents
%cd xp-agents
!pip install -q minigrid

In [ ]:
import os
if not os.path.exists(f'{OUT}/kairi_s{SEED}.json'):
    !python spikes/spike_minigrid.py --train-size 6 --steps $STEPS --seed $SEED --out $OUT/kairi_s$SEED.json
else:
    print('kairi run already on Drive, skipping')

In [ ]:
import os
if not os.path.exists(f'{OUT}/kaiser_s{SEED}.json'):
    !python spikes/spike_minigrid.py --train-size 6 --steps $STEPS --seed $SEED --rnd --out $OUT/kaiser_s$SEED.json
else:
    print('kaiser run already on Drive, skipping')

In [ ]:
import json, pandas as pd
rows = []
for name in ('kairi', 'kaiser'):
    r = json.load(open(f'{OUT}/{name}_s{SEED}.json'))
    t = r['train']
    rows.append({'agent': name, 'env_only_sps': round(r['env_only_sps']), 'train_sps': round(t['sps']), 'train_minutes': round(t['seconds'] / 60, 1), 'steps_to_80pct': t['steps_to_80pct_success'],
                 **{f'size{s}_trained': r['eval_zero_shot'][s]['success'] for s in r['eval_zero_shot']},
                 **{f'size{s}_random': r['random_baseline'][s]['success'] for s in r['random_baseline']}})
pd.set_option('display.width', 200)
summary = pd.DataFrame(rows)
print(summary.T)
print(r['versions'])